# ⚡ Notebook 1: Circuit Breaker — from Bad to Best

When a downstream service is sick, pounding it with more requests **makes things worse**:
- the sick service stays overloaded and can't recover,
- our threads pile up waiting for timeouts,
- callers upstream of us time out too → the failure cascades.

A **circuit breaker** watches failures and, when they cross a threshold, **opens** — short-circuiting the call and returning an error immediately, without touching the downstream.

### Analogy 🏠
Your home's electrical breaker trips so the wires don't melt. Same idea — a fuse for software.

### Three states
```
  +--------+  failures >= threshold   +------+
  | CLOSED | -----------------------> | OPEN |
  +--------+                          +------+
      ^                                  |
      | trial succeeds                   | after cool-down
      |                                  v
  +-----------+   trial fails   +-----------+
  | HALF_OPEN | <-------------- | HALF_OPEN |
  +-----------+                 +-----------+
```
1. `CLOSED` — all good, calls pass through.
2. `OPEN` — too many failures; calls fail instantly (fast fail).
3. `HALF_OPEN` — after a cool-down, allow a trial call. Success → `CLOSED`. Failure → `OPEN` again.

We'll walk **bad → better → best** implementations so you can see *why* each piece exists.

## 🛠️ Setup

```bash
cd 05-microservices/circuit-breaker
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

## 😱 Step 0 — Bad: no breaker

We just call the downstream. Every failure costs us the full timeout. Imagine this is a 1-second timeout across 10 requests — that's 10 seconds of blocked threads, for nothing.

In [ ]:
import time

def sick_service():
    # Pretend the downstream hangs for 0.2s, then errors out.
    time.sleep(0.2)
    raise RuntimeError('downstream down')

def call_no_breaker():
    try:
        return sick_service()
    except Exception as e:
        return f'FAIL: {e}'

t0 = time.time()
results = [call_no_breaker() for _ in range(10)]
print(f'no breaker: {time.time()-t0:.2f}s for 10 reqs — every call paid the timeout')


## 🙂 Step 1 — Naive breaker (counter only)

First try: count failures; after N in a row, **stop calling** and return an error instantly.

**Problem:** once it opens, it never recovers. The downstream could be healthy again, but we never check.


In [ ]:
class NaiveBreaker:
    def __init__(self, fail_threshold=3):
        self.failures = 0
        self.open = False
        self.fail_threshold = fail_threshold

    def call(self, fn, *a, **kw):
        if self.open:
            raise RuntimeError('circuit OPEN — failing fast')
        try:
            return fn(*a, **kw)
        except Exception:
            self.failures += 1
            if self.failures >= self.fail_threshold:
                self.open = True
            raise

cb = NaiveBreaker(fail_threshold=3)
t0 = time.time()
for i in range(10):
    try:
        cb.call(sick_service)
    except Exception as e:
        print(f'  {i}: {e}')
print(f'naive breaker: {time.time()-t0:.2f}s for 10 reqs (fast-fails after 3 failures)')
print(f'breaker stays OPEN forever? {cb.open}  ← the bug: no recovery path')


## 🙂🙂 Step 2 — Better: add `HALF_OPEN` and auto-reset

After a **cool-down**, we let *one trial call* through. If it works, the downstream probably recovered → close the circuit. If it fails, open again.

This is the **classic circuit breaker** pattern.

In [ ]:
class CircuitBreaker:
    def __init__(self, fail_threshold=3, reset_after=1.0):
        self.state = 'CLOSED'
        self.failures = 0
        self.opened_at = 0.0
        self.fail_threshold = fail_threshold
        self.reset_after = reset_after

    def call(self, fn, *a, **kw):
        # If OPEN and enough time has passed, try a single probe.
        if self.state == 'OPEN':
            if time.time() - self.opened_at >= self.reset_after:
                self.state = 'HALF_OPEN'
                print('  → HALF_OPEN (trial call allowed)')
            else:
                raise RuntimeError('circuit OPEN — failing fast')

        try:
            result = fn(*a, **kw)
        except Exception:
            self.failures += 1
            # One failed trial re-opens immediately.
            if self.state == 'HALF_OPEN' or self.failures >= self.fail_threshold:
                self.state = 'OPEN'
                self.opened_at = time.time()
                print('  → OPEN')
            raise

        # Success — close the circuit and reset counter.
        if self.state == 'HALF_OPEN':
            print('  → CLOSED (trial passed)')
        self.state = 'CLOSED'
        self.failures = 0
        return result

def flaky(fail=True):
    if fail:
        raise RuntimeError('downstream down')
    return 'ok'

cb = CircuitBreaker(fail_threshold=3, reset_after=1.0)

print('Phase 1: downstream is broken')
for i in range(6):
    try:
        cb.call(flaky, fail=True)
        print(f'  {i}: ok')
    except Exception as e:
        print(f'  {i}: {e}')

print('\n... waiting out the cool-down ...')
time.sleep(1.1)

print('\nPhase 2: downstream has recovered')
for i in range(3):
    try:
        print(f'  {i}:', cb.call(flaky, fail=False))
    except Exception as e:
        print(f'  {i}: {e}')


## 🏆 Step 3 — Best: time-windowed failures + thread safety

Two subtle real-world issues with Step 2:

1. **Counter never resets over time.** 2 failures today + 1 failure tomorrow shouldn't trip the breaker. Production breakers track failures within a **rolling time window** (e.g. last 10s).
2. **Thread safety.** In a real server many requests run in parallel. State transitions need a lock, or we can get double-opens and weird races.

Here's a tiny windowed, thread-safe version:

In [ ]:
import threading
from collections import deque

class WindowedBreaker:
    """Trips when failures in the last `window_s` seconds exceed `fail_threshold`."""
    def __init__(self, fail_threshold=3, window_s=10.0, reset_after=1.0):
        self.state = 'CLOSED'
        self.failure_times = deque()  # timestamps of recent failures
        self.opened_at = 0.0
        self.fail_threshold = fail_threshold
        self.window_s = window_s
        self.reset_after = reset_after
        self.lock = threading.Lock()

    def _trim(self, now):
        cutoff = now - self.window_s
        while self.failure_times and self.failure_times[0] < cutoff:
            self.failure_times.popleft()

    def call(self, fn, *a, **kw):
        with self.lock:
            now = time.time()
            if self.state == 'OPEN':
                if now - self.opened_at >= self.reset_after:
                    self.state = 'HALF_OPEN'
                else:
                    raise RuntimeError('circuit OPEN — failing fast')

        try:
            result = fn(*a, **kw)
        except Exception:
            with self.lock:
                now = time.time()
                self.failure_times.append(now)
                self._trim(now)
                if self.state == 'HALF_OPEN' or len(self.failure_times) >= self.fail_threshold:
                    self.state = 'OPEN'
                    self.opened_at = now
            raise

        with self.lock:
            self.state = 'CLOSED'
            self.failure_times.clear()
        return result

# Demo: 2 failures, then 3 seconds of nothing, then 1 failure → should NOT trip.
cb = WindowedBreaker(fail_threshold=3, window_s=2.0, reset_after=1.0)
for i in range(2):
    try: cb.call(flaky, fail=True)
    except Exception: pass
print(f'after 2 quick failures: state={cb.state}')

time.sleep(2.1)  # let old failures fall out of the window
try: cb.call(flaky, fail=True)
except Exception: pass
print(f'after window expired + 1 more failure: state={cb.state} (old failures forgotten)')


## 🎯 Key settings to tune

| Parameter | What it controls | Too low | Too high |
|---|---|---|---|
| `fail_threshold` | failures before we trip | flaps on tiny blips | slow to react to outages |
| `window_s` | how far back failures count | forgets real outages | one-off errors add up over days |
| `reset_after` | cool-down before probing | hammers a recovering service | stays open longer than needed |

There is no universal answer — tune to your service's normal error rate and latency.

### What's next
Notebook 2 shows the breaker in action under concurrent load — where it actually saves your system from a cascading failure.